# Unit 5 Hands-On ②: ML-Agents — Pyramids (RND)

버튼을 눌러 피라미드를 소환하고, 피라미드를 쓰러뜨린 뒤 꼭대기의 금 블록으로 이동하는  
에이전트를 **PPO + RND(Random Network Distillation)** 알고리즘으로 훈련합니다.

### SnowballTarget과의 차이점

| 항목 | SnowballTarget | Pyramids |
|---|---|---|
| **난이도** | 낮음 | 높음 |
| **알고리즘** | PPO | PPO + **RND (호기심 기반 탐색)** |
| **훈련 스텝** | 200,000 | **1,000,000** |
| **설정 파일** | 직접 생성 | ML-Agents 내장 파일 수정 |
| **목표 보상** | ≥ 15 | **≥ 1.75** |

### RND(Random Network Distillation)란?
에이전트가 **방문하지 않은 새로운 상태**에 추가 보상을 주는 호기심 기반 탐색 기법입니다.  
Pyramids처럼 희소 보상(버튼 → 피라미드 → 금 블록의 순서를 모두 맞춰야 보상)  
환경에서 일반 PPO보다 훨씬 효과적으로 동작합니다.

---
## 목차
1. 저장소 클론 및 가상 환경 설정
2. ML-Agents 설치
3. Google Drive 마운트
4. Pyramids 환경 다운로드
5. 설정 파일(YAML) 수정
6. 에이전트 훈련
7. 훈련 결과 확인 (TensorBoard)
8. Hugging Face Hub 업로드
9. 브라우저에서 에이전트 플레이 확인


---
## 1. 저장소 클론 및 가상 환경 설정

SnowballTarget 노트북을 이미 실행했다면 이 섹션은 건너뛰어도 됩니다.  
새 Colab 세션이라면 아래 셀을 모두 실행하세요.


In [1]:
# 현재 Python 버전 확인 (ML-Agents는 3.10.12 필요)
!python --version

Python 3.12.13


In [2]:
# ML-Agents 소스코드 클론 (약 2~3분 소요)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

Cloning into 'ml-agents'...
remote: Enumerating objects: 2381, done.
remote: Counting objects: 100% (2381/2381), done.
remote: Compressing objects: 100% (1773/1773), done.
remote: Total 2381 (delta 863), reused 1640 (delta 591), pack-reused 0 (from 0)
Receiving objects: 100% (2381/2381), 97.19 MiB | 26.91 MiB/s, done.
Resolving deltas: 100% (863/863), done.


In [3]:
import os
import sys
import shutil
import subprocess
import urllib.request

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME = 'mlagents'

if 'google.colab' in sys.modules:
    candidate_paths = [
        os.path.join(MINICONDA_DIR, 'bin', 'conda'),
        os.path.expanduser('~/miniconda3/bin/conda'),
        '/usr/local/bin/conda',
    ]
    conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        print('Colab 환경에서 Miniconda를 설치합니다...')
        installer = '/tmp/miniconda.sh'
        urls = [
            'https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh',
            'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        ]

        downloaded = False
        for url in urls:
            try:
                print(f'다운로드 시도: {url}')
                urllib.request.urlretrieve(url, installer)
                downloaded = True
                print('✅ 다운로드 완료')
                break
            except Exception as e:
                print(f'⚠️ 실패: {url} -> {e}')

        if not downloaded:
            raise RuntimeError('Miniconda 설치 파일을 다운로드하지 못했습니다. Colab 네트워크 상태를 확인하세요.')

        result = subprocess.run(
            ['bash', installer, '-b', '-p', MINICONDA_DIR],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        raise RuntimeError('Miniconda 설치가 완료되지 않았습니다. Colab 새로고침 또는 새 런타임으로 다시 시도하세요.')

    print('conda 경로:', conda_path)

    # conda 약관 동의 (필수)
    print('conda 채널 약관을 승인합니다...')
    subprocess.run(
        [conda_path, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/main'],
        capture_output=True,
        text=True,
        check=False,
    )
    subprocess.run(
        [conda_path, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/r'],
        capture_output=True,
        text=True,
        check=False,
    )

    env_check = subprocess.run(
        [conda_path, 'env', 'list'],
        capture_output=True,
        text=True,
        check=False,
    )
    print(env_check.stdout)

    try:
        subprocess.run(
            [conda_path, 'create', '-y', '-n', ENV_NAME, 'python=3.10.12', 'ujson'],
            capture_output=True,
            text=True,
            check=True,
        )
        print('✅ conda 환경 생성 완료')
    except subprocess.CalledProcessError as e:
        print('❌ conda 환경 생성 실패')
        print('STDOUT:')
        print(e.stdout)
        print('STDERR:')
        print(e.stderr)
        if 'already exists' in (e.stdout or '') + (e.stderr or ''):
            print('ℹ️ 이미 같은 이름의 conda 환경이 존재합니다. 기존 환경을 사용합니다.')
        else:
            raise

    print('환경 이름:', ENV_NAME)
else:
    print('현재 노트북은 Colab/Linux 환경이 아닙니다.')
    if shutil.which('conda'):
        print('로컬에 conda가 이미 설치되어 있습니다.')
        print(f'아래 명령으로 환경을 생성하세요: conda create -y -n {ENV_NAME} python=3.10.12 ujson')
    else:
        print('이 노트북은 Google Colab용입니다. 로컬 Windows/VS Code에서는 Miniconda를 먼저 설치한 뒤 다시 실행하세요.')
        print('설치 링크: https://www.anaconda.com/download/success')


Colab 환경에서 Miniconda를 설치합니다...
다운로드 시도: https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh
⚠️ 실패: https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh -> HTTP Error 404: Not Found
다운로드 시도: https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
✅ 다운로드 완료
PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3

conda 경로: /content/miniconda3/bin/conda
conda 채널 약관을 승인합니다...

# conda environments:
#
# * -> active
# + -> frozen
base    

In [4]:
import os
import subprocess

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME = 'mlagents'
conda_path = os.path.join(MINICONDA_DIR, 'bin', 'conda')

if os.path.exists(conda_path):
    result = subprocess.run(
        [conda_path, 'run', '-n', ENV_NAME, 'python', '--version'],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f'conda 실행 파일이 없습니다: {conda_path}')
    print('이전 셀에서 Miniconda 설치가 실패했는지 확인하세요.')


Python 3.10.12


---
## 2. ML-Agents 설치


In [5]:
# conda 환경에 ML-Agents 패키지를 설치합니다.
import os
import subprocess

%cd /content/ml-agents

commands = [
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "--upgrade", "pip"],
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "-e", "./ml-agents-envs"],
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "-e", "./ml-agents"],
]

for cmd in commands:
    print(f"\n>>> {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"명령 실패: {' '.join(cmd)}")

print('✅ ML-Agents 설치 완료')


/content/ml-agents

>>> /content/miniconda3/bin/conda run -n mlagents python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2


>>> /content/miniconda3/bin/conda run -n mlagents python -m pip install -e ./ml-agents-envs
Obtaining file:///content/ml-agents/ml-agents-envs
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# 설치 확인
!mlagents-learn --help | head -5

/bin/bash: line 1: mlagents-learn: command not found


---
## 3. Google Drive 마운트

훈련 결과를 Drive에 저장하여 세션 종료 후에도 보존합니다.

```
Google Drive/RL_Course/Unit5_Pyramids/
└── results/
    └── PyramidsTraining/   ← 훈련 체크포인트 및 최종 모델(.onnx)
```


In [8]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 경로를 원하는 대로 변경하세요.
DRIVE_BASE    = '/content/drive/MyDrive/RL_Course/Unit5_Pyramids'
DRIVE_RESULTS = f'{DRIVE_BASE}/results'

os.makedirs(DRIVE_RESULTS, exist_ok=True)

# 훈련 결과 폴더를 Drive로 심볼릭 링크
if not os.path.exists('/content/ml-agents/results'):
    os.symlink(DRIVE_RESULTS, '/content/ml-agents/results')
    print(f'✅ 심볼릭 링크 생성: /content/ml-agents/results → {DRIVE_RESULTS}')
else:
    print('ℹ️  results 폴더가 이미 존재합니다.')

print(f'결과 저장 경로: {DRIVE_RESULTS}')


Mounted at /content/drive
✅ 심볼릭 링크 생성: /content/ml-agents/results → /content/drive/MyDrive/RL_Course/Unit5_Pyramids/results
결과 저장 경로: /content/drive/MyDrive/RL_Course/Unit5_Pyramids/results


---
## 4. Pyramids 환경 다운로드

Pyramids는 Unity 공식 예제 환경으로, HF Spaces에 호스팅된 파일을 다운로드합니다.


In [9]:
# 환경 실행 파일 저장 디렉토리 생성
!mkdir -p ./training-envs-executables/linux

In [10]:
# Pyramids 환경 다운로드 (HF Spaces에서 제공)
!wget -q "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" \
    -O ./training-envs-executables/linux/Pyramids.zip
print('✅ 다운로드 완료')

✅ 다운로드 완료


In [11]:
# 압축 해제
!unzip -q -d ./training-envs-executables/linux/ \
    ./training-envs-executables/linux/Pyramids.zip
print('✅ 압축 해제 완료')

✅ 압축 해제 완료


In [12]:
# 실행 권한 부여
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids
print('✅ 실행 권한 설정 완료')

✅ 실행 권한 설정 완료


---
## 5. 설정 파일(YAML) 수정

Pyramids는 ML-Agents 내장 설정 파일(`config/ppo/PyramidsRND.yaml`)을 사용합니다.  
기본값에서 **`max_steps`를 1,000,000으로 변경**합니다.

### PyramidsRND.yaml 주요 설정

| 파라미터 | 값 | 설명 |
|---|---|---|
| `trainer_type` | ppo | 알고리즘 |
| `max_steps` | **1,000,000** | 총 훈련 스텝 (기본값보다 줄임) |
| `time_horizon` | 128 | 한 번에 수집하는 스텝 수 |
| `reward_signals.curiosity` | RND | 호기심 보상 (방문하지 않은 상태에 추가 보상) |

> ⏱️ 예상 훈련 시간: **30~45분** (GPU 사용 시) / Pyramids는 Unity Team에서 만들었기 때문에, 기본적으로 yaml이 존재함


In [23]:
import re

config_path = './config/ppo/PyramidsRND.yaml'

# 기존 설정 파일 읽기
with open(config_path, 'r') as f:
    config = f.read()

# max_steps를 1,000,000으로 변경
# 기본값이 너무 커서 Colab 세션 시간 내에 완료하기 어렵기 때문
config_modified = re.sub(r'max_steps:\s*\d+', 'max_steps: 100000', config)

with open(config_path, 'w') as f:
    f.write(config_modified)

print('✅ PyramidsRND.yaml 수정 완료 (max_steps: 100000)')
print('\n현재 설정:')
!cat {config_path}


✅ PyramidsRND.yaml 수정 완료 (max_steps: 100000)

현재 설정:
behaviors:
  Pyramids:
    trainer_type: ppo
    hyperparameters:
      batch_size: 128
      buffer_size: 2048
      learning_rate: 0.0003
      beta: 0.01
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
      rnd:
        gamma: 0.99
        strength: 0.01
        network_settings:
          hidden_units: 64
          num_layers: 3
        learning_rate: 0.0001
    keep_checkpoints: 5
    max_steps: 100000
    time_horizon: 128
    summary_freq: 30000


---
## 6. 에이전트 훈련

```
mlagents-learn <설정파일>      ← PyramidsRND.yaml
    --env=<환경실행파일>        ← Pyramids 실행 파일
    --run-id=<실행ID>          ← 결과 폴더명
    --no-graphics              ← Colab 환경: 렌더링 없이 실행
    --resume                   ← 중단된 훈련 재개 (첫 실행 시 제거)
```

> SnowballTarget(10~35분)보다 훨씬 오래 걸립니다.  
> Colab Pro 사용 시 세션 유지 시간이 길어 안정적으로 훈련할 수 있습니다.


In [30]:
# Pyramids 에이전트 훈련
# Colab 커널이 아닌 conda 환경에서 실행합니다.
!{MINICONDA_DIR}/bin/conda run -n {ENV_NAME} mlagents-learn ./config/ppo/PyramidsRND.yaml \
    --env=./training-envs-executables/linux/Pyramids/Pyramids \
    --run-id="PyramidsTraining" \
    --no-graphics \
    --resume


            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.8.0+cu128
[INFO] Connected to Unity environment with package version 2.2.1-exp.1 and communication version 1.5.0
[INFO] Connected new brain: Pyramids?team=0

---
## 7. 훈련 결과 확인 (TensorBoard)

**주요 지표:**
- `Environment/Cumulative Reward`: 목표 ≥ 1.75
- `Losses/Policy Loss`: 정책 손실 (감소 추세가 좋음)
- `Curiosity/Forward Loss`: RND 호기심 손실 (탐색이 줄어들면 감소)


In [28]:
# TensorBoard 실행
%load_ext tensorboard
%tensorboard --logdir results/PyramidsTraining


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

In [32]:
# 결과 파일 확인
!ls results/PyramidsTraining

configuration.yaml  Pyramids  Pyramids.onnx  run_logs


---
## 8. Hugging Face Hub 업로드

| 파라미터 | 설명 | 예시 |
|---|---|---|
| `--run-id` | 훈련 실행 ID | `PyramidsTraining` |
| `--local-dir` | 로컬 결과 폴더 | `./results/PyramidsTraining` |
| `--repo-id` | HF Hub 저장소 | `username/ppo-Pyramids` |
| `--commit-message` | 커밋 메시지 | `"First Push"` |

### 사전 준비
1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰 입력
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')


In [ ]:
# ✏️ 아래 값을 본인 정보로 수정하세요.
RUN_ID     = 'PyramidsTraining'
LOCAL_DIR  = f'./results/{RUN_ID}'
REPO_ID    = 'YOUR_HF_USERNAME/ppo-PyramidsRND'  # ← username 변경
COMMIT_MSG = 'First Push'

import os
from pathlib import Path

# 업로드에 필요한 설정 파일이 있는 실제 결과 디렉터리 찾기
candidate_dirs = [
    Path(LOCAL_DIR),
    Path('./results') / RUN_ID,
    Path('./results') / RUN_ID / 'run-*',
]

resolved_dir = None
for path in candidate_dirs:
    if path.exists() and (path / 'configuration.yaml').exists():
        resolved_dir = path
        break

if resolved_dir is None:
    matches = sorted(Path('./results').glob(f'{RUN_ID}*')) if Path('./results').exists() else []
    for match in matches:
        if (match / 'configuration.yaml').exists():
            resolved_dir = match
            break

if resolved_dir is None:
    raise FileNotFoundError(f'업로드 가능한 결과 폴더를 찾지 못했습니다: {LOCAL_DIR}')

LOCAL_DIR = str(resolved_dir)
print(f'업로드 대상 경로: {LOCAL_DIR}')

!{MINICONDA_DIR}/bin/conda run -n {ENV_NAME} mlagents-push-to-hf \
    --run-id={RUN_ID} \
    --local-dir={LOCAL_DIR} \
    --repo-id={REPO_ID} \
    --commit-message="{COMMIT_MSG}"

FileNotFoundError: 업로드 가능한 결과 폴더를 찾지 못했습니다: ./results/SnowballTarget1

---
## 9. 브라우저에서 에이전트 플레이 확인

업로드 완료 후 아래 HF Space에서 에이전트 플레이를 실시간으로 확인할 수 있습니다.

1. 아래 링크 접속: https://huggingface.co/spaces/unity/ML-Agents-Pyramids
2. **Step 1**: 본인의 HF username 입력 → 검색
3. **Step 2**: 모델 저장소 선택
4. **Step 3**: `Pyramids.onnx` 선택

> 🎯 목표 점수: **Mean Reward ≥ 1.75**  
> 💡 체크포인트별 모델을 비교하여 훈련 단계별 성능 변화를 확인해보세요.


In [ ]:
print(f'모델 확인: https://huggingface.co/{REPO_ID}')
print(f'플레이 확인: https://huggingface.co/spaces/unity/ML-Agents-Pyramids')
